# 03 - Feature Engineering

## Objetivo

Construir a primeira versão do dataset de modelagem em nível `user_id-product_id` para o sistema de recomendação de próximo carrinho.

Este notebook cobre:

- Separação dos conjuntos `prior` e `train` a partir do dataset unificado.
- Definição de usuários elegíveis (usuários com pedido `train` conhecido).
- Geração de candidatos user-produto com base no histórico `prior` (estratégia v1).
- Construção do target supervisionado usando o pedido `train`.
- Cálculo de features históricas em nível de usuário, produto, user-produto e categoria.
- Tratamento explícito de valores ausentes com documentação de cada decisão.
- Validação anti-leakage do dataset final.
- Persistência do dataset de modelagem em `data/features/`.

## Inputs

- `data/processed/orders_product_unified.parquet`

## Outputs esperados

- `data/features/modeling_dataset_v1.parquet` — dataset de modelagem em nível `user_id-product_id`
- `data/features/features_metadata_v1.json` — dicionário de features com nome, tipo, origem e decisão de imputação

## Limites deste notebook

Este notebook não treina modelos.

Este notebook não implementa baselines.

Este notebook não define a arquitetura MLP.

O conjunto `train` é usado exclusivamente para construção do target supervisionado. Nenhuma feature é calculada a partir de `train`.

O conjunto `test` não está disponível no dataset unificado e não é usado neste notebook.

A estratégia de candidatos adotada aqui é a v1: apenas produtos já comprados pelo usuário no `prior`. Estratégias futuras (populares globais, populares por categoria) serão implementadas em versões posteriores.

A amostra em `data/test_sample/orders_product_sample.parquet` não deve ser usada para conclusões finais. Pode ser usada somente para acelerar o desenvolvimento de código durante a construção do notebook.

---

## 1. Setup inicial

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [3]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TEST_SAMPLE_DIR = DATA_DIR / "test_sample"
FEATURES_DIR = DATA_DIR / "features"

UNIFIED_DATASET_PATH = PROCESSED_DIR / "orders_product_unified.parquet"
SAMPLE_DATASET_PATH = TEST_SAMPLE_DIR / "orders_product_sample.parquet"

OUTPUT_DATASET_PATH = FEATURES_DIR / "modeling_dataset_v1.parquet"
OUTPUT_METADATA_PATH = FEATURES_DIR / "features_metadata_v1.json"

assert UNIFIED_DATASET_PATH.exists(), (
    f"Dataset unificado não encontrado em: {UNIFIED_DATASET_PATH}. "
    "Execute o notebook 01 antes de continuar."
)

print("Arquivos de entrada encontrados com sucesso.")
print(f"Output será salvo em: {FEATURES_DIR}")

Arquivos de entrada encontrados com sucesso.
Output será salvo em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features


### 1.1 Carregamento dos dados

In [4]:
USE_SAMPLE = False

dataset_path = SAMPLE_DATASET_PATH if USE_SAMPLE else UNIFIED_DATASET_PATH

df = pd.read_parquet(dataset_path)

print(f"Dataset carregado: {dataset_path.name}")
print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")

if USE_SAMPLE:
    print(
        "\n⚠️  ATENÇÃO: amostra auxiliar carregada. "
        "Não use este resultado para conclusões finais."
    )

Dataset carregado: orders_product_unified.parquet
Shape: 33,819,106 linhas x 15 colunas


In [5]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 15 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   product_id              int64  
 2   add_to_cart_order       int64  
 3   reordered               int64  
 4   user_id                 int64  
 5   eval_set                object 
 6   order_number            int64  
 7   order_dow               int64  
 8   order_hour_of_day       int64  
 9   days_since_prior_order  float64
 10  product_name            object 
 11  aisle_id                int64  
 12  department_id           int64  
 13  aisle                   object 
 14  department              object 
dtypes: float64(1), int64(10), object(4)
memory usage: 10.6 GB


In [6]:
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


---

## 2. Definição do universo de modelagem

Esta seção estabelece o contrato do dataset supervisionado antes de qualquer feature ser calculada.

Todas as decisões tomadas aqui definem o escopo do problema: quais usuários existem, quais pares `user_id-product_id` são candidatos, e o que significa `target = 1` ou `target = 0`.

Erros nesta seção se propagam silenciosamente para todas as features e para o modelo. Por isso, cada etapa é acompanhada de validação explícita.

### 2.1 Separação prior / train

O dataset unificado contém pedidos dos conjuntos `prior` e `train` identificados pela coluna `eval_set`.

- `prior`: histórico de compras do usuário. Será usado exclusivamente para calcular features.
- `train`: próximo pedido conhecido do usuário. Será usado exclusivamente para construir o target supervisionado.

Nenhuma informação de `train` deve entrar no cálculo de features.

In [7]:
df_prior = df[df["eval_set"] == "prior"].copy()
df_train = df[df["eval_set"] == "train"].copy()

assert not df_prior.empty, "df_prior está vazio. Verifique o dataset unificado."
assert not df_train.empty, "df_train está vazio. Verifique o dataset unificado."
assert set(df_prior["eval_set"].unique()) == {"prior"}, "df_prior contém registros fora do conjunto prior."
assert set(df_train["eval_set"].unique()) == {"train"}, "df_train contém registros fora do conjunto train."

print(f"df_prior: {df_prior.shape[0]:,} linhas | {df_prior['user_id'].nunique():,} usuários | {df_prior['order_id'].nunique():,} pedidos")
print(f"df_train: {df_train.shape[0]:,} linhas | {df_train['user_id'].nunique():,} usuários | {df_train['order_id'].nunique():,} pedidos")

df_prior: 32,434,489 linhas | 206,209 usuários | 3,214,874 pedidos
df_train: 1,384,617 linhas | 131,209 usuários | 131,209 pedidos


### 2.2 Usuários elegíveis

Usuários elegíveis são aqueles com exatamente um pedido `train` conhecido.

O notebook 01 já validou que cada usuário possui no máximo um pedido `train`. Aqui essa condição é reafirmada e o conjunto de usuários elegíveis é materializado para uso nas etapas seguintes.

Usuários com `eval_set = test` não possuem produtos conhecidos no próximo pedido e não fazem parte do dataset de modelagem supervisionado.

In [38]:
train_orders_per_user = (
    df_train.groupby('user_id')['order_id']
    .nunique()
)

assert (train_orders_per_user == 1).all(), (
    "Existem usuários com mais de um pedido train. Verifique o dataset unificado."
)

eligible_users = set(df_train['user_id'].unique())

print(f"Usuários elegíveis (com pedidos train): {len(eligible_users):,}")
print(f"Usuário apenas com prior (sem pedido train): {df_prior['user_id'].nunique() - len(eligible_users):,}")
print(f"Porcentagem de usuário elegíveis: {100 * len(eligible_users) / df_prior['user_id'].nunique()}")

Usuários elegíveis (com pedidos train): 131,209
Usuário apenas com prior (sem pedido train): 75,000
Porcentagem de usuário elegíveis: 63.629133548972156


### 2.3 Geração de candidatos — estratégia v1

**Estratégia adotada**: para cada usuário elegível, os candidatos são os produtos que ele já comprou ao menos uma vez no histórico `prior`.

```txt
candidates(user) = produtos distintos comprados pelo usuário em prior
```

**Por que essa estratégia?**

- É simples, rastreável e sem risco de leakage.
- Está alinhada com o forte sinal de recompra observado no EDA.
- Reduz drasticamente o espaço de candidatos em relação ao produto cartesiano completo.
- Permite validar o pipeline end-to-end antes de adicionar complexidade.

**Limitações conhecidas desta estratégia**

- Não recomenda produtos novos para o usuário.
- Tem um teto de recall estrutural: produtos que aparecem no `train` mas nunca foram comprados no `prior` não podem ser recuperados por esta estratégia.
- Usuários com histórico mais longo tendem a ter mais candidatos, o que pode introduzir viés.

O diagnóstico do teto de recall será feito na seção 2.5.

**Evoluções futuras planejadas** (fora do escopo deste notebook):

```txt
candidates(user) =
produtos comprados pelo usuário no prior ← v1 (este notebook)
+ produtos populares globalmente ← v2
+ produtos populares em aisles/departments preferidos ← v3
+ produtos sugeridos por similaridade entre usuários/produtos ← v4
```


In [20]:
candidates = (
    df_prior[df_prior['user_id'].isin(eligible_users)][['user_id', 'product_id']]
    .drop_duplicates()
    .reset_index(drop = True)
)

assert candidates['user_id'].isin(eligible_users).all(), (
    "Candidatos contêm usuários não elegíveis."
)
assert not candidates.duplicated(subset=["user_id", "product_id"]).any(), (
    "Existem pares user_id-product_id duplicados nos candidatos."
)

print(f"Pares candidatos (user_id-product_id): {len(candidates):,}")
print(f"Usuários cobertos: {candidates['user_id'].nunique():,}")
print(f"Produtos distintos nos candidatos: {candidates['product_id'].nunique():,}")
print(f"Média de candidatos por usuário: {len(candidates) / candidates['user_id'].nunique():,.1f}")

Pares candidatos (user_id-product_id): 8,474,661
Usuários cobertos: 131,209
Produtos distintos nos candidatos: 49,468
Média de candidatos por usuário: 64.6


### 2.4 Construção do target supervisionado

O target é construído comparando os candidatos com os produtos efetivamente comprados no pedido `train` de cada usuário.

```txt
target = 1 → produto candidato aparece no pedido train do usuário
target = 0 → produto candidato não aparece no pedido train do usuário
```

**Sobre os negativos implícitos**

Os exemplos com `target = 0` são chamados de **negativos implícitos**: a ausência de um produto no pedido `train` não significa que o usuário rejeita aquele produto. Pode significar que ele simplesmente não precisava daquele produto naquele momento, que estava fora de estoque, ou que o ciclo de consumo ainda não havia se completado.

Essa limitação é estrutural em sistemas de recomendação baseados em comportamento implícito e é amplamente aceita na literatura e na indústria. O que isso implica na prática:

- O modelo não aprende rejeição explícita, apenas ausência de compra naquele pedido.
- A calibração do modelo pode ser afetada: **scores altos para negativos implícitos não são necessariamente erros.**
- Métricas de avaliação como Precision@K e Recall@K devem ser interpretadas com esse contexto.

Estratégias de **negative sampling** (sortear um subconjunto dos negativos ao invés de usar todos) podem ser exploradas em experimentos futuros para mitigar esse efeito, especialmente em usuários com histórico muito longo.

In [26]:
train_products = (
    df_train[df_train['user_id'].isin(eligible_users)][['user_id', 'product_id']]
    .drop_duplicates()
    .assign(target = 1)
)

modeling_df = candidates.merge(
    train_products,
    on = ['user_id', 'product_id'],
    how = 'left'
)

modeling_df['target'] = modeling_df['target'].fillna(0).astype(int)

assert len(modeling_df) == len(candidates), (
    'O merge com train_products alterou a quantidade de candidatos.'
)
assert modeling_df["target"].isin([0, 1]).all(), (
    "A coluna target contém valores fora do esperado {0, 1}."
)
assert not modeling_df.duplicated(subset=["user_id", "product_id"]).any(), (
    "Existem pares user_id-product_id duplicados no dataset de modelagem."
)

print(f"Dataset de modelagem: {len(modeling_df):,} pares")
print(f"Positivos (target=1): {(modeling_df['target'] == 1).sum():,}")
print(f"Negativos (target=0): {(modeling_df['target'] == 0).sum():,}")

Dataset de modelagem: 8,474,661 pares
Positivos (target=1): 828,824
Negativos (target=0): 7,645,837


### 2.5 Diagnóstico inicial do target

Antes de calcular qualquer feature, é necessário entender a distribuição do target e os limites estruturais da estratégia de candidatos adotada.

Os diagnósticos abaixo respondem três perguntas:

1. **Balanceamento**: qual é a proporção de positivos e negativos?
2. **Distribuição de candidatos por usuário**: os usuários têm tamanhos semelhantes de espaço de candidatos?
3. **Teto de recall (coverage)**: qual é o percentual de produtos do `train` que a estratégia v1 consegue cobrir?

In [31]:
target_dist = (
    modeling_df['target']
    .value_counts()
    .reset_index(name = 'count')
    .assign(pct = lambda d: 100* d['count'] / d['count'].sum())
    .sort_values('target')
)

target_dist

,target,count,pct
0,0,7645837,90.219975
1,1,828824,9.780025


In [34]:
candidates_per_user = (
    modeling_df.groupby('user_id')['product_id']
    .count()
    .rename('candidate_count')
)

candidates_per_user.describe(
    percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).to_frame()

,candidate_count
count,131209.000000
mean,64.589022
std,56.564099
min,1.000000
1%,3.000000
5%,8.000000
25%,25.000000
50%,48.000000
75%,86.000000
95%,179.000000


In [43]:
# Total de produtos no pedido train dos usuários elegíveis
total_train_products = len(train_products)

# Produtos do train que estão cobertos pelos candidatos v1 (target=1)
covered_train_products = (modeling_df["target"]).sum()

# Produtos do train que NÃO estão nos candidatos (não recuperáveis pela estratégia v1)
uncovered_train_products = total_train_products - covered_train_products

recall_ceiling = covered_train_products / total_train_products

print(f"Produtos no pedido train (total):          {total_train_products:,}")
print(f"Cobertos pelos candidatos v1 (target=1):   {covered_train_products:,}")
print(f"Não cobertos — fora da estratégia v1:      {uncovered_train_products:,}")
print(f"\nTeto de recall da estratégia v1:           {recall_ceiling:.2%}")
print(
    f"\n→ {100 - recall_ceiling * 100:.1f}% dos produtos do pedido train "
    "nunca foram comprados pelo usuário no prior e não podem ser recuperados "
    "pela estratégia de candidatos atual."
)

Produtos no pedido train (total):          1,384,617
Cobertos pelos candidatos v1 (target=1):   828,824
Não cobertos — fora da estratégia v1:      555,793

Teto de recall da estratégia v1:           59.86%

→ 40.1% dos produtos do pedido train nunca foram comprados pelo usuário no prior e não podem ser recuperados pela estratégia de candidatos atual.


**Leituras a partir dos diagnósticos**

- O forte desbalanceamento entre positivos e negativos é esperado e reflete a realidade do problema: em cada pedido, o usuário compra uma pequena fração dos produtos que já comprou antes.

- **Distribuição heterogênea de candidatos**: A mediana de 48 candidatos por usuário mascara uma cauda longa significativa. Enquanto 50% dos usuários têm até 48 produtos candidatos, os top 5% têm 179 ou mais, e o outlier máximo atinge 726 candidatos. Usuários com histórico muito longo contribuem com muitos exemplos negativos que podem distorcer o treinamento se não forem tratados com cuidado.

- **Teto de recall estrutural de 59.86% — achado crítico**: De 1.384.617 produtos comprados no pedido train, apenas 828.824 (59.86%) foram comprados antes no prior. Isso significa que **40.14% dos produtos que o usuário compra no train são novos** — produtos que nunca havia comprado antes. Esses produtos são irrecuperáveis pela estratégia v1 de candidatos. Qualquer modelo, por mais bem otimizado que seja, não conseguirá ultrapassar esse teto sem expandir a estratégia de geração de candidatos.